In [16]:
%load_ext autoreload
%autoreload 2

import torch

from utils import create_edge_loaders, create_hetero_graph
from gnn import model_factory
from train import train, evaluate, gnn_evaluate

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
# Follow and activity data paths
FOLLOW_EDGELIST = 'data/higgs-social_network.edgelist'
ACTIVITY_EDGELIST = 'data/higgs-activity_time.txt'
target_interaction = "follow"

torch.manual_seed(42)

In [3]:
model_path = train(
    model_name="sage", 
    target_interaction=target_interaction, 
    activity_data_path=ACTIVITY_EDGELIST, 
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 13640/13640 [02:32<00:00, 89.25it/s, v_num=80, train_loss_step=0.494, train_acc_step=0.759, val_loss=0.596, val_auroc=0.879, val_acc=0.729, val_ap=0.827, val_recall=0.883, train_loss_epoch=0.588, train_acc_epoch=0.727] 

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 13640/13640 [02:32<00:00, 89.25it/s, v_num=80, train_loss_step=0.494, train_acc_step=0.759, val_loss=0.596, val_auroc=0.879, val_acc=0.729, val_ap=0.827, val_recall=0.883, train_loss_epoch=0.588, train_acc_epoch=0.727]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_follow/best-gnn-sage-followepoch=01-val_ap=0.8736.ckpt


In [18]:
# Evaluation
best_model_path = model_path
# best_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt"
print(f"Evaluating best model from: {best_model_path}")

evaluate(
    checkpoint_path=best_model_path,
    model_name="sage",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=0
)

Evaluating best model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/2923 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 2923/2923 [07:13<00:00,  6.75it/s]



--- Standard Metrics ---
Loss:   0.6702
AUROC:  0.8910
AP:     0.8123
Recall: 0.9234

--- Explanation Metrics (over 2923 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.8183 (Higher is better)
Average Fidelity-: 0.8556 (Lower is better)


In [5]:
# Using GNNExplainer
# best_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt"
gnn_evaluate(
    checkpoint_path=best_model_path,
    model_name="sage",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 2923/2923 [00:13<00:00, 217.28it/s]



--- Standard Metrics ---
Loss:  0.6671
AUROC: 0.8922
AP:    0.8241

--- Setting up Explainer (Option B) ---


Explaining:   0%|          | 0/5 [00:00<?, ?it/s]


--- Top Explanations for Target Edge 0 ---
  Top 5 important 'FL' edges:
    User 245 -> User 23 (Weight: 0.1968)
    User 19 -> User 23 (Weight: 0.1922)
    User 2 -> User 1 (Weight: 0.1855)
    User 247 -> User 23 (Weight: 0.1732)
    User 5 -> User 1 (Weight: 0.1676)
  Top 5 important 'RT' edges:
    User 288 -> User 23 (Weight: 0.7907)
    User 274 -> User 5 (Weight: 0.0831)
    User 259 -> User 2 (Weight: 0.0831)
    User 280 -> User 5 (Weight: 0.0830)
    User 281 -> User 5 (Weight: 0.0830)
  Top 5 important 'MT' edges:
    User 69 -> User 4 (Weight: 0.9349)
    User 23 -> User 1 (Weight: 0.1225)
    User 316 -> User 23 (Weight: 0.1030)
    User 317 -> User 23 (Weight: 0.1015)
    User 1 -> User 23 (Weight: 0.0894)
  Top 5 important 'RE' edges:
    User 325 -> User 2 (Weight: 0.9332)
    User 320 -> User 2 (Weight: 0.9331)
    User 315 -> User 23 (Weight: 0.9200)
    User 1 -> User 23 (Weight: 0.9117)
    User 23 -> User 1 (Weight: 0.8642)


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 645 -> User 45 (Weight: 0.1482)
    User 723 -> User 49 (Weight: 0.1472)
    User 51 -> User 1 (Weight: 0.1465)
    User 861 -> User 59 (Weight: 0.1463)
    User 678 -> User 47 (Weight: 0.1462)
  Top 5 important 'RT' edges:
    User 43 -> User 1 (Weight: 0.1525)
    User 914 -> User 31 (Weight: 0.1055)
    User 668 -> User 46 (Weight: 0.1051)
    User 905 -> User 21 (Weight: 0.1049)
    User 1013 -> User 52 (Weight: 0.1049)
  Top 5 important 'MT' edges:
    User 63 -> User 0 (Weight: 0.1100)
    User 62 -> User 0 (Weight: 0.1040)
    User 1045 -> User 10 (Weight: 0.0981)
    User 1048 -> User 24 (Weight: 0.0974)
    User 908 -> User 24 (Weight: 0.0972)
  Top 5 important 'RE' edges:
    User 63 -> User 0 (Weight: 0.0854)
    User 1072 -> User 45 (Weight: 0.0816)
    User 1056 -> User 33 (Weight: 0.0816)
    User 1051 -> User 25 (Weight: 0.0816)
    User 1067 -> User 45 (Weight: 0.0816)


Explaining:  80%|████████  | 4/5 [00:06<00:01,  1.69s/it]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 238 -> User 8 (Weight: 0.1495)
    User 654 -> User 38 (Weight: 0.1480)
    User 689 -> User 123 (Weight: 0.1476)
    User 596 -> User 33 (Weight: 0.1474)
    User 551 -> User 30 (Weight: 0.1468)
  Top 5 important 'RT' edges:
    User 90 -> User 1 (Weight: 0.0922)
    User 71 -> User 1 (Weight: 0.0922)
    User 82 -> User 1 (Weight: 0.0920)
    User 1076 -> User 59 (Weight: 0.0920)
    User 88 -> User 1 (Weight: 0.0920)
  Top 5 important 'MT' edges:
    User 119 -> User 1 (Weight: 0.0882)
    User 100 -> User 1 (Weight: 0.0882)
    User 97 -> User 1 (Weight: 0.0879)
    User 102 -> User 1 (Weight: 0.0877)
    User 98 -> User 1 (Weight: 0.0876)
  Top 5 important 'RE' edges:
    User 123 -> User 1 (Weight: 0.0794)
    User 119 -> User 1 (Weight: 0.0791)
    User 102 -> User 1 (Weight: 0.0791)
    User 118 -> User 1 (Weight: 0.0790)
    User 122 -> User 1 (Weight: 0.0789)


Explaining: 100%|██████████| 5/5 [00:08<00:00,  1.71s/it]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 756 -> User 48 (Weight: 0.1494)
    User 232 -> User 10 (Weight: 0.1488)
    User 365 -> User 19 (Weight: 0.1487)
    User 39 -> User 1 (Weight: 0.1482)
    User 1060 -> User 105 (Weight: 0.1475)
  Top 5 important 'RT' edges:
    User 1137 -> User 21 (Weight: 0.1085)
    User 1144 -> User 31 (Weight: 0.1076)
    User 79 -> User 0 (Weight: 0.1068)
    User 1123 -> User 21 (Weight: 0.1068)
    User 1110 -> User 11 (Weight: 0.1067)
  Top 5 important 'MT' edges:
    User 1214 -> User 11 (Weight: 0.1021)
    User 93 -> User 0 (Weight: 0.1018)
    User 97 -> User 0 (Weight: 0.1018)
    User 1211 -> User 9 (Weight: 0.1017)
    User 1228 -> User 11 (Weight: 0.1017)
  Top 5 important 'RE' edges:
    User 115 -> User 0 (Weight: 0.0853)
    User 110 -> User 0 (Weight: 0.0847)
    User 107 -> User 0 (Weight: 0.0839)
    User 1291 -> User 32 (Weight: 0.0839)
    User 1273 -> User 53 (Weight: 0.0839)

--- Explanation 

In [3]:
# Comparison with vanilla GNN
vanilla_model_path = train(
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 13640/13640 [03:16<00:00, 69.25it/s, v_num=96, train_loss_step=0.515, train_acc_step=0.938, val_loss=0.605, val_auroc=0.789, val_acc=0.855, val_ap=0.699, val_recall=0.590, train_loss_epoch=0.540, train_acc_epoch=0.931]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 13640/13640 [03:16<00:00, 69.25it/s, v_num=96, train_loss_step=0.515, train_acc_step=0.938, val_loss=0.605, val_auroc=0.789, val_acc=0.855, val_ap=0.699, val_recall=0.590, train_loss_epoch=0.540, train_acc_epoch=0.931]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_follow/best-gnn-simple-followepoch=00-val_ap=0.7895.ckpt


In [4]:
print(f"Evaluating vanilla GNN model from: {vanilla_model_path}")
evaluate(
    checkpoint_path=vanilla_model_path,
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=5
)
gnn_evaluate(
    checkpoint_path=vanilla_model_path,
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Evaluating vanilla GNN model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_follow/best-gnn-simple-followepoch=00-val_ap=0.7895.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/2923 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 2923/2923 [09:34<00:00,  5.09it/s]



--- Standard Metrics ---
Loss:   0.6245
AUROC:  0.8685
AP:     0.7743
Recall: 0.7656

--- Explanation Metrics (over 2923 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.7513 (Higher is better)
Average Fidelity-: 0.7249 (Lower is better)
Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 2923/2923 [00:15<00:00, 193.02it/s]



--- Standard Metrics ---
Loss:  0.6245
AUROC: 0.8685
AP:    0.7744

--- Setting up Explainer (Option B) ---


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 634 -> User 64 (Weight: 0.1879)
    User 956 -> User 64 (Weight: 0.1804)
    User 953 -> User 62 (Weight: 0.1743)
    User 206 -> User 62 (Weight: 0.1740)
    User 944 -> User 62 (Weight: 0.1730)
  Top 5 important 'RT' edges:
    User 62 -> User 1 (Weight: 0.3744)
    User 999 -> User 54 (Weight: 0.0988)
    User 1031 -> User 56 (Weight: 0.0988)
    User 967 -> User 33 (Weight: 0.0988)
    User 983 -> User 38 (Weight: 0.0987)
  Top 5 important 'MT' edges:
    User 64 -> User 0 (Weight: 0.1342)
    User 221 -> User 13 (Weight: 0.0947)
    User 63 -> User 0 (Weight: 0.0943)
    User 1060 -> User 33 (Weight: 0.0942)
    User 1109 -> User 56 (Weight: 0.0941)
  Top 5 important 'RE' edges:
    User 64 -> User 0 (Weight: 0.0968)
    User 197 -> User 12 (Weight: 0.0857)
    User 247 -> User 16 (Weight: 0.0850)
    User 1129 -> User 38 (Weight: 0.0849)
    User 1127 -> User 38 (Weight: 0.0848)


Explaining:  80%|████████  | 4/5 [00:05<00:01,  1.41s/it]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 3 -> User 0 (Weight: 0.0000)
    User 2 -> User 0 (Weight: 0.0000)
    User 4 -> User 0 (Weight: 0.0000)
    User 6 -> User 0 (Weight: 0.0000)
    User 5 -> User 0 (Weight: 0.0000)
  Top 5 important 'RT' edges:
    User 63 -> User 1 (Weight: 0.0000)
    User 62 -> User 1 (Weight: 0.0000)
    User 64 -> User 1 (Weight: 0.0000)
    User 66 -> User 1 (Weight: 0.0000)
    User 65 -> User 1 (Weight: 0.0000)
  Top 5 important 'MT' edges:
    User 93 -> User 1 (Weight: 0.0000)
    User 92 -> User 1 (Weight: 0.0000)
    User 94 -> User 1 (Weight: 0.0000)
    User 96 -> User 1 (Weight: 0.0000)
    User 95 -> User 1 (Weight: 0.0000)
  Top 5 important 'RE' edges:
    User 121 -> User 1 (Weight: 0.0000)
    User 95 -> User 1 (Weight: 0.0000)
    User 122 -> User 1 (Weight: 0.0000)
    User 119 -> User 1 (Weight: 0.0000)
    User 100 -> User 1 (Weight: 0.0000)

--- Top Explanations for Target Edge 4 ---
  Top 5 impor

Explaining: 100%|██████████| 5/5 [00:27<00:00,  5.46s/it]


--- Explanation Metrics (over 5 edges) ---
Average Fidelity+: 0.0000 (Higher is better)
Average Fidelity-: 0.0000 (Lower is better)


In [5]:
# Ablation study: stripped GNN
stripped_model_path = train(
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_follow exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                   | Params | Mode  | FLOPs
-

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 13640/13640 [02:41<00:00, 84.56it/s, v_num=104, train_loss_step=0.540, train_acc_step=0.889, val_loss=0.609, val_auroc=0.720, val_acc=0.812, val_ap=0.623, val_recall=0.443, train_loss_epoch=0.550, train_acc_epoch=0.886]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 13640/13640 [02:41<00:00, 84.55it/s, v_num=104, train_loss_step=0.540, train_acc_step=0.889, val_loss=0.609, val_auroc=0.720, val_acc=0.812, val_ap=0.623, val_recall=0.443, train_loss_epoch=0.550, train_acc_epoch=0.886]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_follow/best-gnn-stripped-followepoch=00-val_ap=0.6863.ckpt


In [15]:
print(f"Evaluating stripped GNN model from: {stripped_model_path}")
evaluate(
    checkpoint_path=stripped_model_path,
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=5
)
gnn_evaluate(
    checkpoint_path=stripped_model_path,
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Evaluating stripped GNN model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_follow/best-gnn-stripped-followepoch=00-val_ap=0.6863.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/2923 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 2923/2923 [04:47<00:00, 10.16it/s]



--- Standard Metrics ---
Loss:   0.6071
AUROC:  0.7545
AP:     0.6648
Recall: 0.5157

--- Explanation Metrics (over 2923 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.4680 (Higher is better)
Average Fidelity-: 0.4420 (Lower is better)
Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 2923/2923 [00:17<00:00, 171.76it/s]



--- Standard Metrics ---
Loss:  0.6069
AUROC: 0.7545
AP:    0.6648

--- Setting up Explainer (Option B) ---


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 58 -> User 1 (Weight: 0.9157)
    User 51 -> User 1 (Weight: 0.9113)
    User 40 -> User 1 (Weight: 0.9107)
    User 49 -> User 1 (Weight: 0.9075)
    User 32 -> User 1 (Weight: 0.8965)
  Top 5 important 'RT' edges:
    User 62 -> User 1 (Weight: 0.1134)
    User 876 -> User 7 (Weight: 0.0000)
    User 874 -> User 2 (Weight: 0.0000)
    User 877 -> User 7 (Weight: 0.0000)
    User 875 -> User 6 (Weight: 0.0000)
  Top 5 important 'MT' edges:
    User 64 -> User 0 (Weight: 0.9287)
    User 63 -> User 0 (Weight: 0.0729)
    User 968 -> User 4 (Weight: 0.0000)
    User 3 -> User 3 (Weight: 0.0000)
    User 967 -> User 3 (Weight: 0.0000)
  Top 5 important 'RE' edges:
    User 64 -> User 0 (Weight: 0.0878)
    User 968 -> User 4 (Weight: 0.0000)
    User 967 -> User 3 (Weight: 0.0000)
    User 575 -> User 35 (Weight: 0.0000)
    User 968 -> User 4 (Weight: 0.0000)


Explaining:  80%|████████  | 4/5 [00:03<00:00,  1.16it/s]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 45 -> User 1 (Weight: 0.0880)
    User 61 -> User 1 (Weight: 0.0879)
    User 59 -> User 1 (Weight: 0.0877)
    User 23 -> User 0 (Weight: 0.0876)
    User 49 -> User 1 (Weight: 0.0876)
  Top 5 important 'RT' edges:
    User 71 -> User 1 (Weight: 0.0800)
    User 90 -> User 1 (Weight: 0.0799)
    User 88 -> User 1 (Weight: 0.0799)
    User 82 -> User 1 (Weight: 0.0799)
    User 84 -> User 1 (Weight: 0.0799)
  Top 5 important 'MT' edges:
    User 97 -> User 1 (Weight: 0.0800)
    User 100 -> User 1 (Weight: 0.0800)
    User 119 -> User 1 (Weight: 0.0800)
    User 102 -> User 1 (Weight: 0.0800)
    User 120 -> User 1 (Weight: 0.0799)
  Top 5 important 'RE' edges:
    User 123 -> User 1 (Weight: 0.0751)
    User 120 -> User 1 (Weight: 0.0750)
    User 115 -> User 1 (Weight: 0.0750)
    User 118 -> User 1 (Weight: 0.0749)
    User 119 -> User 1 (Weight: 0.0749)


Explaining: 100%|██████████| 5/5 [00:04<00:00,  1.16it/s]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 39 -> User 1 (Weight: 0.9263)
    User 50 -> User 1 (Weight: 0.1146)
    User 37 -> User 1 (Weight: 0.1028)
    User 47 -> User 1 (Weight: 0.0998)
    User 32 -> User 1 (Weight: 0.0982)
  Top 5 important 'RT' edges:
    User 76 -> User 0 (Weight: 0.0808)
    User 71 -> User 0 (Weight: 0.0806)
    User 79 -> User 0 (Weight: 0.0806)
    User 60 -> User 0 (Weight: 0.0805)
    User 78 -> User 0 (Weight: 0.0805)
  Top 5 important 'MT' edges:
    User 97 -> User 0 (Weight: 0.0801)
    User 92 -> User 0 (Weight: 0.0800)
    User 72 -> User 0 (Weight: 0.0800)
    User 99 -> User 0 (Weight: 0.0800)
    User 102 -> User 0 (Weight: 0.0800)
  Top 5 important 'RE' edges:
    User 113 -> User 0 (Weight: 0.0738)
    User 72 -> User 0 (Weight: 0.0733)
    User 111 -> User 0 (Weight: 0.0732)
    User 1202 -> User 32 (Weight: 0.0000)
    User 411 -> User 26 (Weight: 0.0000)

--- Explanation Metrics (over 5 edges) ---
Aver